Index(['Guid', 'Straat', 'Wegsectie', 'Dofunctie', 'Verhardingstype',
       'Wegfunctie', 'Oppervlakte', 'Wegsectieonderdeel_id',
       'Structurele index', 'Structurele index_date', 'Globale staat',
       'Globale staat_date', 'Visuele index', 'Gemeente', 'DATUM',
       'Prioriteit', 'k', 'j'],

In [1]:
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Any
import datetime


class WegVakonderdeel:
    """Dataklasse voor een wegvakonderdeel volgens OCW-systematiek"""

    def __init__(self, **kwargs):
        self.guid = kwargs.get('guid')
        self.straat = kwargs.get('straat')
        self.oppervlakte =  kwargs.get('oppervlakte') # in meters2
        self.verharding = kwargs.get('verharding')  # 'asfalt', 'beton', 'elementen'
        self.functie = kwargs.get('functie')  # 'erf', 'verzamel', 'doorgang'
        self.leeftijd = self.bereken_leeftijd(kwargs.get('structurele_index_date', 0))
        self.visuele_index = kwargs.get('visuele_index', 0)
        self.visuele_index_date = kwargs.get('visuele_index_date')
        self.structurele_index = kwargs.get('structurele_index', 0)
        self.structurele_index_date = kwargs.get('structurele_index_date')
        self.globale_index = kwargs.get('globale_index', 0.5 * (self.structurele_index+self.visuele_index))
        self.globale_index_date = kwargs.get('globale_index_date')
        self.toegewezen_strategie = kwargs.get('toegewezen_strategie', 1)
        # Track maintenance actions (original dict format)
        self.onderhouds_acties = {}
        self.uitgebreid_model_jaar = kwargs.get('X', 0)
        self.df_uitgebreid_model = pd.DataFrame(columns=["CRF", "CE", "CEA", "CEAC", "CEACE",
                     "CR", "CRA", "CRACC", "CTA", "disconteringsvoet",
                    "inflatie", "sigma", "reparatie_kost", "jaarlijks_onderhoud"
                     ])

        # Degradation/maintenance history as DataFrame
        self.df_onderhouds_historie = pd.DataFrame(columns=[
            'jaar', 'scenario_nm', 'visueel_index', 'structureel_index',
            'globaal_index', 'onderhoud_type', 'cumul_B', 'cumul_W', 'B_value', 'cost'
        ])

    def bereken_leeftijd(self, datum):
        vandaag = datetime.datetime.now()
        return vandaag.year - datum.year - ((vandaag.month, vandaag.day) < (datum.month, datum.day))

    def set_uitgebreid_model(self, new_data, jaar):
        self.uitgebreid_model_jaar = jaar
        self.df_uitgebreid_model = new_data
        self.df_uitgebreid_model.index = [f"{year}" for year in range(self.visuele_index_date.year, self.visuele_index_date.year +30)]
        self.df_uitgebreid_model.columns.name = "year"

    def set_onderhoud(self, data: dict, scenario_nm: int): #FORMAT DICT='jaar': 0, 'visueel_index': visueel,'structureel_index': structureel,'globaal_index': segment.globale_index,'onderhoud_type': onderhoud_type
        if scenario_nm not in self.onderhouds_acties:
            self.onderhouds_acties[scenario_nm] = {}

        self.onderhouds_acties[scenario_nm] = data
        new_data = pd.DataFrame(data)

        if self.df_onderhouds_historie.empty:
            self.df_onderhouds_historie = new_data
        else:
            self.df_onderhouds_historie = pd.concat(
                [self.df_onderhouds_historie, new_data],
                ignore_index=True
            )

    def __repr__(self):
        return (
            f"WegVakonderdeel(guid={self.guid!r}, straat={self.straat!r}, oppervlakte={self.oppervlakte}, "
            f"verharding={self.verharding!r}, functie={self.functie!r}, leeftijd={self.leeftijd}, "
            f"visuele_index={self.visuele_index}, structurele_index={self.structurele_index}, "
            f"globale_index={self.globale_index}, globale_index_date={self.globale_index_date})"
            f"scenario={self.onderhouds_acties}"
        )


In [2]:
file = "../Destelbergen_wso_excel_20230509.xlsx"
df = pd.read_excel(file)
df.DATUM = pd.to_datetime(df['DATUM'])
df['Structurele index_date'] = pd.to_datetime(df['Structurele index_date'])
df['Globale staat_date'] = pd.to_datetime(df['Globale staat_date'])

# Map road functions (k)
road_function_map = {
    'ERF': 'erf',  # Erffunctie
    'VW': 'verzamel',   # Verzamelweg
    'DGW': 'doorgang'   # Doorgangsweg
}

# Map pavement types (j)
pavement_type_map = {
    'BS': 'asfalt',  # Asfalt (bitumineuze verharding)
    'BP': 'beton',  # Beton
    'CS': 'elementen'   # Elementenverharding
}

# Create new columns with mapped values
df['functie'] = df['Wegfunctie'].map(road_function_map)
df['verharding'] = df['Verhardingstype'].map(pavement_type_map)

print(df.shape)
df.info()

(2286, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2286 entries, 0 to 2285
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Guid                    2286 non-null   object        
 1   Straat                  2286 non-null   object        
 2   Wegsectie               2286 non-null   object        
 3   Dofunctie               2286 non-null   object        
 4   Verhardingstype         2286 non-null   object        
 5   Wegfunctie              2286 non-null   object        
 6   Oppervlakte             2286 non-null   float64       
 7   Wegsectieonderdeel_id   2286 non-null   int64         
 8   Structurele index       2286 non-null   float64       
 9   Structurele index_date  2270 non-null   datetime64[ns]
 10  Globale staat           2286 non-null   float64       
 11  Globale staat_date      2270 non-null   datetime64[ns]
 12  Visuele index           2286 non-null

/var/folders/ln/gbs9zy0n4fs374gdtp6rp1xw0000gn/T/ipykernel_37163/3712089176.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df.DATUM = pd.to_datetime(df['DATUM'])


In [3]:
import os
import json

class OCWSystematiek:
    def __init__(self):
        script_dir = os.path.dirname(os.getcwd())
        self._load_configurations(script_dir)
        self._validate_parameters()

    def _load_configurations(self, config_dir):
        """Load all configurations from JSON files"""
        # Load main parameters
        with open(os.path.join(config_dir, '_parameters.json'), 'r') as f:
            params = json.load(f)
            self.drempels = params['drempels']
            self.economie = params['economie']
            self.prijzen = params['prijzen']
        
        # Load strategies
        with open(os.path.join(config_dir, '_strategieen.json'), 'r') as f:
            self.strategieen = json.load(f)
        
        # Load combined parameters
        with open(os.path.join(config_dir, '_wegkenmerken.json'), 'r') as f:
            combined = json.load(f)
            self.wegkenmerken = combined['wegkenmerken']
            self.BW_standaard = combined['BW_standaard']
            self.onderhoud_B = combined['onderhoud_B']

    def _validate_parameters(self):
        """Valideer of alle parameters geldige waarden hebben"""
        # Valideer drempelwaarden (moeten tussen 0 en 1 liggen)
        for naam, waarde in self.drempels.items():
            assert 0 <= waarde <= 1, f"Ongeldige drempelwaarde voor {naam}: {waarde}"

        # Valideer economische parameters (moeten positief zijn)
        for econ_type, params in self.economie.items():
            assert params['r'] > 0, f"Ongeldige disconteringsfactor r in {econ_type}"
            assert params['i'] >= 0, f"Ongeldige prijsindex i in {econ_type}"
            assert params['sigma'] >= 0, f"Ongeldige sigma in {econ_type}"

        # Valideer prijzen (moeten niet-negatief zijn)
        for verharding, prijslijst in self.prijzen.items():
            for onderhoud_type, prijs in prijslijst.items():
                assert prijs >= 0, (
                    f"Negatieve prijs voor {verharding}/{onderhoud_type}: {prijs}"
                )

        # Controleer consistentie tussen drempels
        assert self.drempels['routine'] > self.drempels['lokaal'], (
            "Routine drempel moet hoger zijn dan lokale reparaties"
        )
        assert self.drempels['lokaal'] > self.drempels['algemeen'], (
            "Lokale drempel moet hoger zijn dan algemene reparaties"
        )
        assert self.drempels['algemeen'] > self.drempels['versterking'], (
            "Algemene drempel moet hoger zijn dan versterking"
        )

        # Log bevestiging
        print("Validatie van parameters succesvol voltooid!")

    def bereken_index(self, weg_type, verharding, dienstjaren, onderhoudstype=None):
        K1 = self.wegkenmerken[weg_type]['K1']
        K2 = self.wegkenmerken[weg_type]['K2']
        T = self.wegkenmerken[weg_type]['T']
        W = self.BW_standaard[verharding]['W']

        #.get(onderhoudstype, fallback)
        B = self.onderhoud_B[verharding].get(onderhoudstype, self.BW_standaard[verharding]['B']) 

        visueel_index, structureel_index, globaal_index = [], [], []

        for jaar in range(1, dienstjaren + 1):
            I = 0.9 - K1 * (1 + T) * B * jaar
            U = 0.9 - K2 * (1 + T) * W * jaar
            G = (I + U) / 2
            visueel_index.append(I)
            structureel_index.append(U)
            globaal_index.append(G)

        return visueel_index, structureel_index, globaal_index

    def _check_technische_leeftijd(self, segment: WegVakonderdeel) -> bool:
        """
        Controleert of reconstructie nodig is op basis van technische leeftijd
        volgens wegtype-specifieke maxima:
        - Asfalt: 25 jaar
        - Beton: 40 jaar
        - Elementen: 30 jaar
        """
        max_leeftijd = {
            'asfalt': 25,
            'beton': 40,
            'elementen': 30
        }.get(segment.verharding, 30)

        return segment.leeftijd >= max_leeftijd

    def bereken_index_segment(self, segment: WegVakonderdeel, jaren: int, 
                             scenario_nm: int=18) -> List[Dict]:
        """
        Calculate index evolution with cumulative degradation and dynamic B-values.
        Key improvements:
        1. Tracks cumulative degradation separately for visual (B) and structural (W) indices
        2. Dynamically adjusts B-value based on maintenance type
        3. Applies road-type specific degradation models
        4. Properly resets degradation after maintenance
        """
        K1 = self.wegkenmerken[segment.functie]['K1']
        K2 = self.wegkenmerken[segment.functie]['K2']
        T = self.wegkenmerken[segment.functie]['T']
        W = self.BW_standaard[segment.verharding]['W']
        current_B = self.BW_standaard[segment.verharding]['B'] # Start with standard B value (will change if maintenance occurs)
        
        # Track cumulative degradation
        cumul_B = max(0, (0.9 - segment.visuele_index))  # Existing degradation
        cumul_W = max(0, (0.9 - segment.structurele_index))
        data = []
        
        # Huidige staat (mogelijk al gedegradeerd)
        # Initialize state
        visueel = segment.visuele_index
        structureel = segment.structurele_index
        globaal = segment.globale_index
        strategie = self.strategieen[str(scenario_nm)]
        onderhoud_type = None
        
        
        # Voeg technische levensduur check toe
        # Handle backlog (immediate maintenance if IG < 0.3)
        if globaal < 0.3 or self._check_technische_leeftijd(segment):
            visueel = structureel = 0.9
            cumul_B = cumul_W = 0
            onderhoud_type = "versterking_v3"
            current_B = self.onderhoud_B[segment.verharding].get(
                    onderhoud_type,
                    self.BW_standaard[segment.verharding]['B'])
            
        # Voeg jaar 0 toe
        data.append({
                'jaar': 0, 'scenario_nm': scenario_nm, 'onderhoud_type': onderhoud_type,
                'visueel_index': visueel, 'structureel_index': structureel,
                'globaal_index': globaal, 'cumul_B': 0, 'cumul_W': 0,
                'B_value': current_B, 'cost': self._bereken_kosten(segment, onderhoud_type)
            })
        
        for jaar in range(1, jaren + 1):
            # Bereken nieuwe degradatie (afhankelijk van wegtype)
            delta_B = K1 * (1 + T) * current_B
            delta_W = K2 * (1 + T) * W
            
            # Pas cumulatieve degradatie toe
            cumul_B += delta_B
            cumul_W += delta_W
            
            # Wegtype-specifieke degradatie (Hoofdstuk 4.4)
            if segment.verharding == 'beton':
                visueel = max(0, 0.9 - cumul_B * np.sqrt(jaar))  # Vierkantswortel-model voor beton
            else:
                visueel = max(0, 0.9 - cumul_B)  # Lineair model voor asfalt/elementen
            
            structureel = max(0, 0.9 - cumul_W)
            globaal = (visueel + structureel) / 2
            
            # Bepaal onderhoudsadvies
            onderhoud_type = self._bepaal_onderhoudsadvies_new(strategie, globaal, data[-1]['globaal_index'])
            
            # Log huidige staat
            data.append({
                'jaar': jaar, 'scenario_nm': scenario_nm, 'visueel_index': visueel, 'structureel_index': structureel,
                'globaal_index': globaal,'onderhoud_type': onderhoud_type,'cumul_B': cumul_B, 'cumul_W': cumul_W,
                'B_value': current_B, 'cost': self._bereken_kosten(segment, onderhoud_type)
            })
            
            # Pas onderhoud toe (reset cumulatieve degradatie indien nodig)
            if onderhoud_type:
                # 1. First update the B-value for future degradation calculations
                current_B = self.onderhoud_B[segment.verharding].get(
                    onderhoud_type,
                    self.BW_standaard[segment.verharding]['B']
                )
                
                # 2. Apply maintenance effects
                (visueel, structureel, 
                cumul_B, cumul_W) = self._pas_index_aan_na_onderhoud(
                                            onderhoud_type=onderhoud_type,
                                            visuele_index=visueel,
                                            structurele_index=structureel,
                                            cumul_B=cumul_B,
                                            cumul_W=cumul_W,
                                            verharding=segment.verharding
                                            )
                

        # BEWAAR ONDERHOUD DATA VOOR SCENARIO
        segment.set_onderhoud(data, scenario_nm)


        return data
    
    def bereken_index_alle_segmenten(self, scenario, segmenten):
        resultaten = {}
        for segment in segmenten:
            resultaten[segment] = self.bereken_index_segment(segment=segment, scenario_nm=scenario, jaren=20)
        return resultaten

    def segmenteren_wegennet(self, df) -> List[WegVakonderdeel]:
        instances = [
            WegVakonderdeel(
                guid=row['Guid'],
                straat=row['Straat'],
                oppervlakte=row['Oppervlakte'],
                verharding=row['verharding'],
                functie=row['functie'],
                visuele_index=row['Visuele index'],
                visuele_index_date=row['DATUM'],
                structurele_index=row['Structurele index'],
                structurele_index_date = row['Structurele index_date'],
                globale_index=row['Globale staat'],
                globale_index_date=row['Globale staat_date']
            )
            for _, row in df.iterrows()
        ]
        return instances

    def voorspel_evolutie(self, segment: WegVakonderdeel, jaren: int) -> List[float]:
        """
        Voorspel evolutie globale index met het bereken_index model uit OCWEvolutieWet.
        """

        # Pas bereken_index toe
        _, _, globaal_index = self.bereken_index(
            weg_type=segment.functie,
            verharding=segment.verharding,
            dienstjaren=jaren,
        )

        return globaal_index
    
    def _bepaal_onderhoudsadvies_new(self, strategie, G, G_min1year):
        onderhoudstypes = ['versterking_v3', 'versterking_v2', 'versterking_v1', 
                           'algemeen_v3', "algemeen_v2", 'algemeen_v1' ]
        for onderhoudstype in onderhoudstypes:
            if (strategie[onderhoudstype] is not None 
            and  G_min1year >= strategie[onderhoudstype]
            and G <= strategie[onderhoudstype]):
                return str(onderhoudstype)
        # EXCEPTION FOR 'lokaal'
        if strategie['lokaal'] is not None and G < strategie['lokaal'] and G > 0.5:
            return 'lokaal'
        
        return None
    
    def _pas_index_aan_na_onderhoud(self, onderhoud_type: str, visuele_index: float,
                             structurele_index: float, cumul_B: float,
                             cumul_W: float, verharding: str) -> Tuple[float, float, float, float]:
        """
        Pas indices aan na onderhoud volgens OCW-richtlijnen.
        Verbeteringen:
        - Wegtype-specifieke herstelfactoren
        - Realistischere indexaanpassingen gebaseerd op onderhoudsintensiteit
        - Differentiatie tussen kortdurend en langdurig effect
        """
        # Basisherstelwaarden (afhankelijk van wegtype)
        effect_params = {
            'asfalt': {
                'lokaal': {'vis_boost': 0.15, 'B_reduction': 0.3},
                'algemeen': {'vis_set': 0.9, 'B_reset': True},
                'versterking': {'full_reset': True}
            },
            'beton': {
                'lokaal': {'vis_boost': 0.10, 'B_reduction': 0.2},
                'algemeen': {'vis_set': 0.85, 'B_reset': True},
                'versterking': {'full_reset': True}
            },
            'elementen': {
                'lokaal': {'vis_boost': 0.05, 'B_reduction': 0.1},
                'algemeen': {'vis_set': 0.8, 'B_reset': True},
                'versterking': {'full_reset': True}
            }
        }
        
        params = effect_params[verharding]
        maint_category = onderhoud_type.split('_')[0]  # Extract 'lokaal', 'algemeen', etc.
    
        # Bepaal herstelfactoren
        # Apply maintenance effects
        if maint_category == 'versterking':
            return (0.9, 0.9, 0.0, 0.0)  # Full reset
        elif maint_category == 'algemeen':
            new_cumul_B = 0.0 if params['algemeen']['B_reset'] else cumul_B
            return (
                    params['algemeen']['vis_set'], structurele_index,
                    new_cumul_B, cumul_W
                )
        elif maint_category == 'lokaal':
            new_vis = min(0.9, visuele_index + params['lokaal']['vis_boost'])
            reduced_B = cumul_B * (1 - params['lokaal']['B_reduction'])
            return (
                    new_vis, structurele_index,
                    reduced_B, cumul_W
                )
        
        return (visuele_index, structurele_index, cumul_B, cumul_W)
    
    def economische_optimalisatie(self, segment: WegVakonderdeel, 
                                onderhoud_type: str) -> Dict:
        """
        Economische optimalisatie voor onderhoud (9.3 en 9.4)
        """
        # Vereenvoudigd model (9.3.4)
        prijs_onderhoud = self.prijzen[segment.verharding][onderhoud_type]
        prijs_lokaal = self.prijzen[segment.verharding]['lokaal']
        T_vereenvoudigd = np.sqrt(prijs_onderhoud / prijs_lokaal)
        print(T_vereenvoudigd)
        
        # Uitgebreid model (9.4.1.2)
        T_uitgebreid = self._bereken_uitgebreid_model(
            segment, onderhoud_type)
        
        return {
            'vereenvoudigd': T_vereenvoudigd,
            'uitgebreid': T_uitgebreid,
            'strategie_advies': self._bepaal_strategie_advies(segment)
        }

    def _bereken_uitgebreid_model(self, segment: WegVakonderdeel, onderhoud_type: str) -> float:
        """Optimaliseer onderhoudstijdstip volgens de exacte Excel-formules."""
        # Configuratie
        r = self.economie['uitgebreid']['r']       # Disconteringsvoet
        i = self.economie['uitgebreid']['i']       # Inflatie
        sigma = self.economie['uitgebreid']['sigma'] # Groeipercentage gewoon onderhoud
        jaren = 30  # Maximale horizon voor analyse

        # Vector van jaren [1, 2, ..., 30]
        t = np.arange(1, jaren + 1)
        
        # Basis kosten (per m² × oppervlakte)
        cr = self.prijzen[segment.verharding][onderhoud_type] * segment.oppervlakte  # Reparatiekost
        ce = self.prijzen[segment.verharding]['lokaal'] * segment.oppervlakte         # Jaarlijks onderhoud

        # 1. Capital Recovery Factor
        CRF = (r * (1 + r)**t) / ((1 + r)**t - 1)
        
        # 2. Gewoon onderhoud (CE)
        CE = ce * (1 + sigma)**(t-1) * (1 + i)**(t-1)  # Groei met sigma en inflatie
        CEA = CE * (1 + r)**-(t-1)                     # Geactualiseerd
        CEAC = np.cumsum(CEA)                          # Cumulatief geactualiseerd
        CEACE = CRF * CEAC                              # Equivalente annuïteit

        # 3. Reparatiekosten (CR)
        CR = cr * (1 + i)**(t-1)                       # Inflatiecorrectie
        CRA = CR * (1 + r)**-(t-1)                     # Geactualiseerd
        CRACC = CRF * CRA                               # Equivalente annuïteit

        # 4. Totale jaarlijkse kost
        CTA = CEACE + CRACC

        disconteringsvoet, inflatie, sigma_array, reparatie_kost, onderhoud_kost = [
            np.full(30, val) for val in (r, i, sigma, cr, ce)
            ]
        columns = (["CRF", "CE", "CEA", "CEAC", "CEACE",
                     "CR", "CRA", "CRACC", "CTA", "disconteringsvoet",
                    "inflatie", "sigma", "reparatie_kost", "jaarlijks_onderhoud"
                     ])
        data = np.column_stack([CRF, CE, CEA, CEAC, CEACE, CR, 
                                CRA, CRACC, CTA, disconteringsvoet, inflatie, 
                                sigma_array, reparatie_kost, onderhoud_kost])
        new_data= pd.DataFrame(data=data, columns=columns)
        segment.set_uitgebreid_model(new_data, np.argmin(CTA) + 1,)

        return np.argmin(CTA) + 1

    def _bepaal_strategie_advies(self, segment: WegVakonderdeel) -> Dict:
        """Bepaal beste strategie op basis van wegkenmerken"""
        # Vereenvoudigde implementatie - kan uitgebreid worden TODO:
        if segment.functie == 'doorgang':
            return {'strategie': 1, 'reden': 'Optimale balans voor doorgangswegen'}
        elif segment.verharding == 'beton':
            return {'strategie': 5, 'reden': 'Aangepaste strategie voor beton'}
        else:
            return {'strategie': 2, 'reden': 'Standaard strategie'}

    def genereer_onderhoudsplan(self, segmenten: List[WegVakonderdeel], 
                              budget: float, strategie_id: int) -> Dict:
        """
        Genereer onderhoudsplan voor het hele netwerk (4.8)
        """
        plan = {j: [] for j in range(20)}  # 5-jarenplan
        
        # Rangschik segmenten op urgentie
        segmenten_gesorteerd = sorted(
            segmenten, 
            key=lambda x: (x.globale_index, -x.leeftijd)
        )
        
        # Verdeel budget over jaren
        for segment in segmenten_gesorteerd:
            advies = self.bepaal_onderhoudsadvies(segment)
            kosten = self._bereken_kosten(segment, advies['type'])
            
            # Zoek eerste jaar met voldoende budget
            for j in range(20):
                if kosten <= budget / 20:  # Gelijk verdelen
                    plan[j].append({
                        'id': segment.id,
                        'type': advies['type'],
                        'kosten': kosten,
                        'voor_IG': segment.globale_index,
                        'verwacht_IG': 0.9  # Na onderhoud
                    })
                    break
                    
        return plan

    def _bereken_kosten(self, segment: WegVakonderdeel, onderhoud_type: str) -> float:
        """Bereken geschatte kosten voor onderhoud"""

        # Get base price per m²
        try:
            prijs_per_m2 = self.prijzen[segment.verharding][onderhoud_type]
        except KeyError:
            prijs_per_m2 = 0

        if onderhoud_type == None:
            return 0
        elif onderhoud_type == 'lokaal':  # lokaal
            return prijs_per_m2 * segment.oppervlakte  # TODO: 10% van oppervlakte
        elif onderhoud_type.startswith(('versterking', 'algemeen')):
            return prijs_per_m2 * segment.oppervlakte
        else:
            return 0  # Geen kosten voor routineonderhoud
    
    def _bereken_rendement(self, segment: WegVakonderdeel, onderhoud_advies: Dict) -> float:
        """Calculate cost-effectiveness of maintenance."""
        if onderhoud_advies['kosten'] <= 0:
            return float('inf')
        return (onderhoud_advies['levensduur_verlenging'] * segment.oppervlakte) / onderhoud_advies['kosten']

    def genereer_netwerkplan(self, segmenten: List[WegVakonderdeel], budget: float, horizon_jaren: int = 20) -> Dict[int, List]:
        """
        Genereert een geoptimaliseerd onderhoudsplan voor het hele netwerk
        met budgetallocatie gebaseerd op:
        - Urgentie (laagste globale index eerst)
        - Wegfunctie (doorgang > verzamel > erf)
        - Kosten-effectiviteit
        """
        # Prioritering: combinatie van urgentie en wegfunctie
        gewichten = {'doorgang': 3, 'verzamel': 2, 'erf': 1}
        
        gesorteerde_segmenten = sorted(
            segmenten,
            key=lambda x: (x.globale_index, -gewichten[x.functie], x.oppervlakte),
        )

        plan = {jaar: [] for jaar in range(horizon_jaren)}
        rest_budget = budget

        for segment in gesorteerde_segmenten:
            advies = self.bepaal_onderhoudsadvies(segment)
            print(f"{segment.guid} {segment.globale_index}- ADVIES {advies}")
            kosten = self._bereken_kosten(segment, advies['type'])

            # Vind eerste jaar met voldoende budget
            for jaar in range(horizon_jaren):
                print(f"jaar {jaar}")
                if kosten <= rest_budget / (horizon_jaren - jaar):
                    plan[jaar].append({
                        'segment_guid': segment.guid,
                        'type': advies['type'],
                        'kosten': kosten,
                        'verwacht_rendement': self._bereken_rendement(segment, advies)
                    })
                    rest_budget -= kosten
                    break

        return plan
    
    def genereer_rapport(self, segmenten: List[WegVakonderdeel]) -> Dict:
        """
        Genereert een rapport met:
        - Prioriteringslijst
        - Budgetbehoefte
        - Verwachte levensduurverlenging
        """
        return {
            'prioriteiten': self._genereer_prioriteitenlijst(segmenten),
            'budget_analyse': self._analyseer_budgetbehoefte(segmenten),
            'levensduur_analyse': self._analyseer_levensduur(segmenten)
        }
    
    def bepaal_onderhoudsadvies(self, segment: WegVakonderdeel) -> Dict[str, Any]:
        """
        Bepaalt het optimale onderhoudsadvies voor een segment, rekening houdend met:
        - Huidige staat (IG)
        - Technische levensduur
        - Kosten-effectiviteit
        - Netwerkbrede prioriteiten
        Retourneert dict met type, urgentie en verwachte levensduurverlenging
        """
        # Controleer technische levensduur eerst
        if self._check_technische_leeftijd(segment):
            return {
                'type': 'versterking_v3',
                'urgentie': 'hoog',
                'reden': 'Technische levensduur overschreden',
                'kosten': self._bereken_kosten(segment, 'versterking_v3'),
                'levensduur_verlenging': 25  # Jaren
            }

        # Bepaal advies op basis van globale index en strategie
        strategie = self.strategieen[str(segment.toegewezen_strategie)]
        onderhoudstypes = ['versterking_v3', 'versterking_v2', 'algemeen_v3', 'algemeen_v2', 'lokaal']
        
        for onderhoud_type in onderhoudstypes:
            drempel = strategie.get(onderhoud_type)
            if drempel and segment.globale_index < drempel:
                # Bereken kosten-effectiviteit
                kosten = self._bereken_kosten(segment, onderhoud_type)
                levensduur_verlenging = self._bereken_levensduur_verlenging(segment, onderhoud_type)
                
                return {
                    'type': onderhoud_type,
                    'urgentie': self._bepaal_urgentie(onderhoud_type, segment.functie),
                    'reden': f"IG < {drempel:.2f}",
                    'kosten': kosten,
                    'kosten_effectiviteit': levensduur_verlenging / kosten if kosten > 0 else float('inf'),
                    'levensduur_verlenging': levensduur_verlenging
                }

        # Standaard geval (routineonderhoud)
        return {
            'type': 'routine',
            'urgentie': 'laag',
            'reden': 'Geen onderhoud nodig',
            'kosten': 0,
            'levensduur_verlenging': 0
        }

    def _bereken_levensduur_verlenging(self, segment: WegVakonderdeel, onderhoud_type: str) -> int:
        """Schat levensduurverlenging in jaren gebaseerd op onderhoudstype"""
        basis_levensduur = {
            'versterking_v3': 25,
            'versterking_v2': 20,
            'algemeen_v3': 12,
            'algemeen_v2': 10,
            'lokaal': 3
        }.get(onderhoud_type, 0)
        
        # Correctie voor wegtype
        return int(basis_levensduur * {
            'asfalt': 1.0,
            'beton': 1.2,
            'elementen': 0.9
        }.get(segment.verharding, 1.0))

    def _bepaal_urgentie(self, onderhoud_type: str, wegfunctie: str) -> str:
        """Bepaalt urgentieniveau voor sortering"""
        type_gewicht = {
            'versterking_v3': 3,
            'versterking_v2': 2,
            'versterking_v1': 1.75,
            'algemeen_v3': 1.5,
            'algemeen_v2': 1,
            'algemeen_v1': 0.9,
            'lokaal': 0.5
        }.get(onderhoud_type, 0)
        
        functie_gewicht = {
            'doorgang': 3,
            'verzamel': 2,
            'erf': 1
        }.get(wegfunctie, 1)
        
        totaal = type_gewicht * functie_gewicht
        
        if totaal >= 6: return 'kritiek'
        elif totaal >= 3: return 'hoog'
        elif totaal >= 1.5: return 'medium'
        return 'laag'
    
    def bereken_totale_kost_voor_scenario_per_segment(self, segment: WegVakonderdeel, scenario_nm: int) -> float:
        """
        Calculate total maintenance costs for a specific scenario
        based on the maintenance actions stored in the segment.
        
        Args:
            segment: The road segment
            scenario_nm: Scenario number to calculate costs for
            
        Returns:
            Total cost for this scenario in euros
        """
        if scenario_nm not in segment.onderhouds_acties:
            return 0.0
        
        onderhouds_data = segment.onderhouds_acties[scenario_nm]
        totaal_kosten = 0.0
        
        for jaar_data in onderhouds_data:
            if jaar_data['onderhoud_type'] and jaar_data['onderhoud_type'] != 'None':
                onderhoud_type = jaar_data['onderhoud_type']
                
                # Get price per m² from your price table
                prijs_per_m2 = self.prijzen[segment.verharding].get(onderhoud_type, 0)
                
                # Calculate cost based on maintenance type
                if onderhoud_type == 'lokaal':
                    # Local repairs affect ~10% of surface
                    kosten = prijs_per_m2 * segment.oppervlakte * 0.1
                elif onderhoud_type.startswith(('algemeen', 'versterking')):
                     # Full surface treatment
                    kosten = prijs_per_m2 * segment.oppervlakte
                else:
                    kosten = 0
                
                totaal_kosten += kosten
        
        return totaal_kosten

    

In [4]:
model = OCWSystematiek()

ocw = model.segmenteren_wegennet(df)

#Segmenteren wegennet
straat_erf_elementen = ocw[200]
straat_erf_asfalt = ocw[20]
straat_doorgang_asfalt = ocw[0]
straat = ocw[998]

print(f"Aantal straten {len(ocw)}")
print(f"Eerste straat: {straat_erf_elementen}")
print(f"Globale index eerste straat: {ocw[0].globale_index:.2f}")
print(f"Globale index tweede straat: {straat.globale_index:.2f}, {straat.functie} {straat.verharding}")

Validatie van parameters succesvol voltooid!
Aantal straten 2286
Eerste straat: WegVakonderdeel(guid='217FEFE6-1EE7-4626-BC93-37020F43878B', straat='7917E921-3252-4ED2-A0E6-1F68FF540958', oppervlakte=521.6672306060791, verharding='elementen', functie='erf', leeftijd=75, visuele_index=0.44013448, structurele_index=0.34816138, globale_index=0.39414793, globale_index_date=1950-01-01 00:00:00)scenario={}
Globale index eerste straat: 0.45
Globale index tweede straat: 0.64, erf asfalt


In [5]:
print(straat)

WegVakonderdeel(guid='11351C7A-B0FA-4615-B08A-FFA7BA5A9877', straat='B5573FF6-E7C9-4D13-B650-1E193E55F270', oppervlakte=425.42480087280273, verharding='asfalt', functie='erf', leeftijd=24, visuele_index=0.89315137, structurele_index=0.396, globale_index=0.64457569, globale_index_date=2001-01-01 00:00:00)scenario={}


In [6]:
#model.voorspel_evolutie(straat,20)
model.bereken_index_alle_segmenten(segmenten=ocw, scenario=3)
model.bereken_index_alle_segmenten(segmenten=ocw, scenario=1)
model.bereken_index_alle_segmenten(segmenten=ocw, scenario=18)

model.prijzen

{'asfalt': {'routine': 0,
  'lokaal': 5.4,
  'algemeen_v1': 50,
  'algemeen_v2': 50,
  'algemeen_v3': 50,
  'versterking_v1': 118,
  'versterking_v2': 118,
  'versterking_v3': 118},
 'beton': {'routine': 0,
  'lokaal': 4.9,
  'algemeen_v1': 87,
  'algemeen_v2': 87,
  'algemeen_v3': 87,
  'versterking_v1': 145,
  'versterking_v2': 145,
  'versterking_v3': 145},
 'elementen': {'routine': 0,
  'lokaal': 10.4,
  'algemeen_v1': 99,
  'algemeen_v2': 99,
  'algemeen_v3': 99,
  'versterking_v1': 140,
  'versterking_v2': 140,
  'versterking_v3': 140}}

In [7]:

totaal_kosten = model.bereken_totale_kost_voor_scenario_per_segment(straat, 3)
print(f"Total maintenance costs for scenario 1: €{totaal_kosten:.2f}")


Total maintenance costs for scenario 1: €115851.68


In [8]:
print(straat_doorgang_asfalt.df_onderhouds_historie[straat.df_onderhouds_historie['scenario_nm'] == 3])
#print(straat.df_onderhouds_historie)

    jaar  scenario_nm  onderhoud_type  visueel_index  structureel_index  \
0      0            3  versterking_v3       0.900000           0.900000   
1      1            3            None       0.871920           0.866304   
2      2            3            None       0.843840           0.832608   
3      3            3            None       0.815760           0.798912   
4      4            3          lokaal       0.787680           0.765216   
5      5            3          lokaal       0.744156           0.731520   
6      6            3          lokaal       0.713689           0.697824   
7      7            3          lokaal       0.692362           0.664128   
8      8            3          lokaal       0.677434           0.630432   
9      9            3          lokaal       0.666984           0.596736   
10    10            3          lokaal       0.659669           0.563040   
11    11            3          lokaal       0.654548           0.529344   
12    12            3    

In [9]:
plan = model.genereer_netwerkplan(ocw, 100000)
print(plan)

975C89D2-3AFF-4A41-9CAD-B2A53673BE4C 0.0- ADVIES {'type': 'versterking_v3', 'urgentie': 'hoog', 'reden': 'Technische levensduur overschreden', 'kosten': 5342.330669403076, 'levensduur_verlenging': 25}
jaar 0
jaar 1
jaar 2
1DB3AE65-2721-4CCD-B358-E2E1CC8D3D95 0.0- ADVIES {'type': 'versterking_v3', 'urgentie': 'hoog', 'reden': 'Technische levensduur overschreden', 'kosten': 26732.78062438965, 'levensduur_verlenging': 25}
jaar 0
jaar 1
jaar 2
jaar 3
jaar 4
jaar 5
jaar 6
jaar 7
jaar 8
jaar 9
jaar 10
jaar 11
jaar 12
jaar 13
jaar 14
jaar 15
jaar 16
jaar 17
E902ABAD-E71C-4AE6-B904-1381D55E61D4 0.0- ADVIES {'type': 'versterking_v3', 'urgentie': 'hoog', 'reden': 'Technische levensduur overschreden', 'kosten': 48038.20314025879, 'levensduur_verlenging': 25}
jaar 0
jaar 1
jaar 2
jaar 3
jaar 4
jaar 5
jaar 6
jaar 7
jaar 8
jaar 9
jaar 10
jaar 11
jaar 12
jaar 13
jaar 14
jaar 15
jaar 16
jaar 17
jaar 18
jaar 19
FBDE6A58-9832-46C6-B805-5074FC172A04 0.0- ADVIES {'type': 'versterking_v3', 'urgentie': 'hoo

In [10]:
print(straat)

WegVakonderdeel(guid='11351C7A-B0FA-4615-B08A-FFA7BA5A9877', straat='B5573FF6-E7C9-4D13-B650-1E193E55F270', oppervlakte=425.42480087280273, verharding='asfalt', functie='erf', leeftijd=24, visuele_index=0.89315137, structurele_index=0.396, globale_index=0.64457569, globale_index_date=2001-01-01 00:00:00)scenario={3: [{'jaar': 0, 'scenario_nm': 3, 'onderhoud_type': None, 'visueel_index': 0.89315137, 'structureel_index': 0.396, 'globaal_index': 0.64457569, 'cumul_B': 0, 'cumul_W': 0, 'B_value': 0.02, 'cost': 0}, {'jaar': 1, 'scenario_nm': 3, 'visueel_index': 0.87315137, 'structureel_index': 0.372, 'globaal_index': 0.6225756849999999, 'onderhoud_type': 'lokaal', 'cumul_B': 0.026848630000000023, 'cumul_W': 0.528, 'B_value': 0.02, 'cost': 2297.293924713135}, {'jaar': 2, 'scenario_nm': 3, 'visueel_index': 0.826205959, 'structureel_index': 0.348, 'globaal_index': 0.5871029795, 'onderhoud_type': 'lokaal', 'cumul_B': 0.07379404100000002, 'cumul_W': 0.552, 'B_value': 0.055, 'cost': 2297.29392471

In [11]:

print(straat.df_onderhouds_historie.loc[straat.df_onderhouds_historie['scenario_nm'] == 18, ['scenario_nm','jaar', 'cost', 'globaal_index','visueel_index', 'onderhoud_type']])

    scenario_nm  jaar  cost  globaal_index  visueel_index onderhoud_type
42           18     0   0.0       0.644576       0.893151           None
43           18     1   0.0       0.622576       0.873151           None
44           18     2   0.0       0.600576       0.853151           None
45           18     3   0.0       0.578576       0.833151           None
46           18     4   0.0       0.556576       0.813151           None
47           18     5   0.0       0.534576       0.793151           None
48           18     6   0.0       0.512576       0.773151           None
49           18     7   0.0       0.490576       0.753151           None
50           18     8   0.0       0.468576       0.733151           None
51           18     9   0.0       0.446576       0.713151           None
52           18    10   0.0       0.424576       0.693151           None
53           18    11   0.0       0.402576       0.673151           None
54           18    12   0.0       0.380576       0.

In [12]:
print(straat_erf_asfalt.df_onderhouds_historie.loc[straat_erf_asfalt.df_onderhouds_historie['scenario_nm'] == 3, ['jaar', 'cost', 'globaal_index','visueel_index', 'onderhoud_type']])

    jaar          cost  globaal_index  visueel_index onderhoud_type
0      0      0.000000       0.828000       0.900000           None
1      1      0.000000       0.806000       0.880000           None
2      2   1592.317873       0.784000       0.860000         lokaal
3      3   1592.317873       0.750500       0.817000         lokaal
4      4   1592.317873       0.723450       0.786900         lokaal
5      5   1592.317873       0.700915       0.765830         lokaal
6      6   1592.317873       0.681540       0.751081         lokaal
7      7   1592.317873       0.664378       0.740757         lokaal
8      8   1592.317873       0.648765       0.733530         lokaal
9      9   1592.317873       0.634235       0.728471         lokaal
10    10   1592.317873       0.620465       0.724930         lokaal
11    11   1592.317873       0.607225       0.722451         lokaal
12    12   1592.317873       0.594358       0.720715         lokaal
13    13   1592.317873       0.581750       0.71

In [13]:
print(straat_erf_elementen.df_onderhouds_historie.loc[straat_erf_elementen.df_onderhouds_historie['scenario_nm'] == 3, ['jaar', 'cost', 'globaal_index','visueel_index', 'onderhoud_type']])

    jaar          cost  globaal_index  visueel_index  onderhoud_type
0      0  73033.412285       0.394148       0.900000  versterking_v3
1      1      0.000000       0.882400       0.884000            None
2      2      0.000000       0.864800       0.868000            None
3      3      0.000000       0.847200       0.852000            None
4      4      0.000000       0.829600       0.836000            None
5      5      0.000000       0.812000       0.820000            None
6      6   5425.339198       0.794400       0.804000          lokaal
7      7   5425.339198       0.781600       0.797600          lokaal
8      8   5425.339198       0.769120       0.791840          lokaal
9      9   5425.339198       0.756928       0.786656          lokaal
10    10   5425.339198       0.744995       0.781990          lokaal
11    11   5425.339198       0.733296       0.777791          lokaal
12    12   5425.339198       0.721806       0.774012          lokaal
13    13   5425.339198       0.710

In [14]:
asvalt_instances = [straat for straat in ocw if straat.verharding == 'asfalt']
asvalt_instances[0]

WegVakonderdeel(guid='BD0E3CD9-7747-4D61-B1A8-3CBF976421E3', straat='78086F54-2CEE-4077-99C8-35A275EF5C05', oppervlakte=1113.1310024261475, verharding='asfalt', functie='doorgang', leeftijd=73, visuele_index=0.9, structurele_index=0.0, globale_index=0.45, globale_index_date=1952-01-01 00:00:00)scenario={3: [{'jaar': 0, 'scenario_nm': 3, 'onderhoud_type': 'versterking_v3', 'visueel_index': 0.9, 'structureel_index': 0.9, 'globaal_index': 0.45, 'cumul_B': 0, 'cumul_W': 0, 'B_value': 0.02, 'cost': 131349.4582862854}, {'jaar': 1, 'scenario_nm': 3, 'visueel_index': 0.87192, 'structureel_index': 0.866304, 'globaal_index': 0.869112, 'onderhoud_type': None, 'cumul_B': 0.028080000000000004, 'cumul_W': 0.033696000000000004, 'B_value': 0.02, 'cost': 0}, {'jaar': 2, 'scenario_nm': 3, 'visueel_index': 0.84384, 'structureel_index': 0.832608, 'globaal_index': 0.8382240000000001, 'onderhoud_type': None, 'cumul_B': 0.05616000000000001, 'cumul_W': 0.06739200000000001, 'B_value': 0.02, 'cost': 0}, {'jaar'

In [15]:
# 6. Economische optimalisatie
optimalisatie = model.economische_optimalisatie(straat_erf_asfalt, onderhoud_type='versterking_v3')
print(f"- Economisch optimale timing: \n{optimalisatie} \nverharding: {straat_erf_asfalt.verharding} - functie: {straat_erf_asfalt.functie}")

    


4.674596437325029
- Economisch optimale timing: 
{'vereenvoudigd': np.float64(4.674596437325029), 'uitgebreid': np.int64(13), 'strategie_advies': {'strategie': 2, 'reden': 'Standaard strategie'}} 
verharding: asfalt - functie: erf


In [16]:
straat_erf_asfalt.df_uitgebreid_model


year,CRF,CE,CEA,CEAC,CEACE,CR,CRA,CRACC,CTA,disconteringsvoet,inflatie,sigma,reparatie_kost,jaarlijks_onderhoud
2022,1.040000,1592.317873,1592.317873,1592.317873,1656.010588,34795.094254,34795.094254,36186.898024,37842.908611,0.04,0.03,0.09,34795.094254,1592.317873
2023,0.530196,1787.695276,1718.937765,3311.255638,1755.614754,35838.947081,34460.526040,18270.835767,20026.450521,0.04,0.03,0.09,34795.094254,1592.317873
2024,0.360349,2007.045486,1855.626374,5166.882011,1861.878385,36914.115494,34129.174828,12298.398294,14160.276679,0.04,0.03,0.09,34795.094254,1592.317873
2025,0.275490,2253.309967,2003.184356,7170.066367,1975.281909,38021.538958,33801.009685,9311.841692,11287.123600,0.04,0.03,0.09,34795.094254,1592.317873
2026,0.224627,2529.791100,2162.476035,9332.542402,2096.342061,39162.185127,33475.999977,7519.617246,9615.959307,0.04,0.03,0.09,34795.094254,1592.317873
2027,0.190762,2840.196468,2334.434466,11666.976867,2225.614704,40337.050681,33154.115361,6324.542122,8550.156826,0.04,0.03,0.09,34795.094254,1592.317873
2028,0.166610,3188.688575,2520.066899,14187.043766,2363.697858,41547.162201,32835.325791,5470.680891,7834.378749,0.04,0.03,0.09,34795.094254,1592.317873
2029,0.148528,3579.940663,2720.460680,16907.504446,2511.234981,42793.577067,32519.601504,4830.065910,7341.300891,0.04,0.03,0.09,34795.094254,1592.317873
2030,0.134493,4019.199382,2936.789620,19844.294066,2668.918497,44077.384379,32206.913028,4331.604119,7000.522616,0.04,0.03,0.09,34795.094254,1592.317873
2031,0.123291,4512.355146,3170.320872,23014.614938,2837.493609,45399.705911,31897.231172,3932.639753,6770.133362,0.04,0.03,0.09,34795.094254,1592.317873


In [4]:
import os
os.listdir(os.getcwd())

['test_panel_global_tab.ipynb',
 'test_input_data.ipynb',
 '__init__.py',
 'model1.ipynb',
 'test_geo.ipynb',
 'test_scenario_class.ipynb',
 'test_uitgebreid_plan.ipynb']